In [0]:
from pyspark.sql import functions as F

MIN_SALE_AMOUNT = 10000            # sales below this are treated as gifts/family transfers
RES_CLASSES = [
    "Res Condo Common Area",
    "Residential Condominium",
    "Residential Duplex",
    "Residential SFR",
    "Residential Triplex",
    "Townhome",
]

bronze = spark.table("gwinnett.bronze.property_ownership_raw")
latest_q = bronze.agg(F.max("source_quarter")).first()[0]   # the newest quarter loaded

parcels = (bronze
    .filter(F.col("source_quarter") == latest_q)            # only process the newest download
    .filter(F.col("PROPCLAS_DESC") == "R")                  # residential group
    .filter(F.trim(F.col("PCDESC")).isin(RES_CLASSES))      # only the property types you chose
)
print(latest_q, parcels.count())